<a href="https://colab.research.google.com/github/sulthonarifimadudin/Midterm_ML/blob/main/fraud_detection_midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/UTS ML/train_transaction.csv')

In [ ]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df.shape

(590540, 394)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB


In [ ]:
df['isFraud'].value_counts()

,count
isFraud,
0,569877
1,20663


In [ ]:
df = df.sample(50000, random_state=42)

In [ ]:
df.shape

(50000, 394)

In [ ]:
df = df.fillna(-999)

In [ ]:
df.select_dtypes(include='object').columns

Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1',
       'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'],
      dtype='object')

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    solver='liblinear'
)

lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98      9641
           1       0.77      0.14      0.23       359

    accuracy                           0.97     10000
   macro avg       0.87      0.57      0.61     10000
weighted avg       0.96      0.97      0.96     10000



In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

In [ ]:
print(classification_report(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99      9641
           1       0.87      0.27      0.42       359

    accuracy                           0.97     10000
   macro avg       0.92      0.64      0.70     10000
weighted avg       0.97      0.97      0.97     10000



In [ ]:
roc_auc_score(y_test, pred_rf)

NameError: name 'roc_auc_score' is not defined

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc_score(y_test, pred_rf)

np.float64(0.6357123230955075)

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.5 MB/s eta 0:00:00


In [ ]:
import optuna

In [ ]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 50, 150)

    max_depth = trial.suggest_int('max_depth', 5, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    score = roc_auc_score(y_test, preds)

    return score

In [ ]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=10)

[I 2026-05-12 07:40:55,331] A new study created in memory with name: no-name-4f071f17-3a60-4bca-b8ce-f12fdbae525c
[I 2026-05-12 07:41:16,455] Trial 0 finished with value: 0.6232293659940614 and parameters: {'n_estimators': 97, 'max_depth': 18}. Best is trial 0 with value: 0.6232293659940614.
[I 2026-05-12 07:41:34,419] Trial 1 finished with value: 0.6025454484517868 and parameters: {'n_estimators': 147, 'max_depth': 9}. Best is trial 0 with value: 0.6232293659940614.
[I 2026-05-12 07:41:40,748] Trial 2 finished with value: 0.615028405553233 and parameters: {'n_estimators': 62, 'max_depth': 13}. Best is trial 0 with value: 0.6232293659940614.
[I 2026-05-12 07:41:57,166] Trial 3 finished with value: 0.6233330896741777 and parameters: {'n_estimators': 146, 'max_depth': 17}. Best is trial 3 with value: 0.6233330896741777.
[I 2026-05-12 07:42:01,636] Trial 4 finished with value: 0.6068793069524625 and parameters: {'n_estimators': 69, 'max_depth': 10}. Best is trial 3 with value: 0.623333089

In [ ]:
study.best_params

{'n_estimators': 95, 'max_depth': 20}

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=95,
    max_depth=20,
    random_state=42
)

best_rf.fit(X_train, y_train)

best_pred = best_rf.predict(X_test)

In [ ]:
print(classification_report(y_test, best_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99      9641
           1       0.88      0.28      0.42       359

    accuracy                           0.97     10000
   macro avg       0.93      0.64      0.70     10000
weighted avg       0.97      0.97      0.97     10000



In [ ]:
roc_auc_score(y_test, best_pred)

np.float64(0.638601562095958)

In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.7/887.7 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import mlflow

In [ ]:
with mlflow.start_run():

    best_rf = RandomForestClassifier(
        n_estimators=95,
        max_depth=20,
        random_state=42
    )

    best_rf.fit(X_train, y_train)

    preds = best_rf.predict(X_test)

    auc = roc_auc_score(y_test, preds)

    mlflow.log_param("n_estimators", 95)
    mlflow.log_param("max_depth", 20)

    mlflow.log_metric("roc_auc", auc)

    print("ROC-AUC:", auc)

2026/05/12 07:47:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/12 07:47:52 INFO mlflow.store.db.utils: Updating database tables


ROC-AUC: 0.638601562095958
